# Vision Transformer (ViT)

### From image patches to classification

A Vision Transformer applies the Transformer architecture to images. This notebook follows a single image through patch extraction, token embeddings, self-attention, and classification. Small PyTorch examples show how the tensor shapes change.

The final section connects the core ViT idea to our selected [ViTAEv2 paper](https://doi.org/10.1007/s11263-022-01739-w).

## 1. What is ViT?

A **Vision Transformer (ViT)** is an image classification model. Instead of processing the entire image as one input to the Transformer, it first divides the image into small patches. Each patch becomes a token in a sequence.

The main path is **image → patches → embeddings → Transformer encoder → class prediction**. This is similar to a language Transformer receiving a sequence of word tokens, but here the tokens represent image regions.

## 2. Split the image into patches

Imagine a **32 × 32** RGB image divided into **8 × 8** squares. There are 4 patches across and 4 down, so the image produces **16 patches**. Each patch contains **8 × 8 × 3 = 192** pixel values.

Patch size controls the sequence length: smaller patches preserve more detail, but create more tokens and require more attention computation. The code below reshapes an image into its 16 flattened patches.

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)
image = torch.rand(1, 3, 32, 32)  # [batch, RGB channels, height, width]
patches = nn.functional.unfold(image, kernel_size=8, stride=8)
patches = patches.transpose(1, 2)
print("Image:", tuple(image.shape))
print("16 flattened patches:", tuple(patches.shape))  # [1, 16, 192]

## 3. Turn patches into embeddings

A raw patch contains pixel values, while the Transformer operates on vectors of a chosen size. A **learned linear projection** maps every flattened patch to an embedding. In our example, each patch changes from **192 values to 64 values**.

The projection uses the same learned transformation for all patches. Its weights are trained along with the rest of the model, so the embeddings become useful for classification.

In [ ]:
projection = nn.Linear(192, 64)
tokens = projection(patches)
print("Patch embeddings:", tuple(tokens.shape))  # [1, 16, 64]

## 4. Add position information

After converting patches into tokens, the Transformer needs to know their original locations. Without position information, shuffling the patch tokens would make it difficult to distinguish different spatial arrangements.

A **position embedding** is added to each token. Its role is simple: patch content describes *what* appears, while position helps describe *where* it appears. The small example below uses learnable position vectors.

## 5. Add a CLS token

ViT places an extra learnable **classification token**, called **CLS**, before the patch tokens. It starts as a model parameter, then exchanges information with the image patches through attention.

After the Transformer encoder, the final CLS representation is sent to a classification layer. Our 16 patch tokens therefore become a sequence of **17 tokens**: one CLS token followed by 16 image tokens.

In [ ]:
cls_token = nn.Parameter(torch.zeros(1, 1, 64))
position = nn.Parameter(torch.zeros(1, 17, 64))
sequence = torch.cat([cls_token.expand(tokens.size(0), -1, -1), tokens], dim=1)
sequence = sequence + position
print("CLS + 16 patches:", tuple(sequence.shape))  # [1, 17, 64]

## 6. Let patches communicate with self-attention

Each token is transformed into three vectors: **Query (Q)**, **Key (K)**, and **Value (V)**. A query is compared with the keys of other tokens. The resulting weights determine how much information to take from their values.

The operation is **Attention(Q, K, V) = softmax(QKᵀ / √d) V**. For example, a patch containing part of an object can attend to a distant patch containing another part. Attention weights are learned through training; they are not predefined rules.

## 7. Use multi-head attention

A single attention head produces one set of relationships between tokens. **Multi-head attention** learns several such sets in parallel and combines their outputs.

Different heads can focus on different visual relationships, although their roles are learned rather than assigned in advance. The output still has one updated representation for each token.

## 8. Pass tokens through a Transformer block

A ViT encoder block contains **multi-head attention** and a **feed-forward network (MLP)**. Attention exchanges information *between* tokens. The MLP then transforms the representation of *each* token.

**Layer normalization** and **residual connections** help the block train. ViT stacks multiple blocks so that information can be refined repeatedly before classification.

## 9. Make the prediction

Once the tokens pass through the encoder, ViT takes the final **CLS token** and sends it to a linear classification head. If the task has 10 classes, the head returns **10 logits**: one score for each class.

The highest score gives the predicted class. During training, the logits are compared with the correct label to calculate the loss.

## 10. See the full path in code

The following small PyTorch model puts the pieces together. A convolution with kernel size and stride equal to the patch size efficiently performs the patch projection. The model then adds CLS and position embeddings, applies two Transformer blocks, and produces class scores.

It is initialized with random weights, so the example demonstrates the **forward pass and tensor shapes**, not meaningful image recognition.

In [ ]:
class TinyViT(nn.Module):
    def __init__(self, image_size=32, patch_size=8, dim=64, num_classes=10):
        super().__init__()
        count = (image_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(3, dim, patch_size, stride=patch_size)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.position = nn.Parameter(torch.randn(1, count + 1, dim) * 0.02)
        block = nn.TransformerEncoderLayer(
            d_model=dim, nhead=4, dim_feedforward=dim * 2,
            batch_first=True, activation="gelu", dropout=0.0
        )
        self.encoder = nn.TransformerEncoder(block, num_layers=2)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls.expand(x.size(0), -1, -1), x], dim=1)
        x = self.encoder(x + self.position)
        return self.head(x[:, 0])

model = TinyViT().eval()
with torch.no_grad():
    scores = model(image)
print("Input:", tuple(image.shape))
print("Class scores:", tuple(scores.shape))  # [1, 10]

## 11. How does ViT learn?

A training set provides images and correct class labels. For each batch, ViT produces logits; a loss function measures the prediction error. Backpropagation updates the patch projection, position embeddings, CLS token, encoder, and classifier.

ViT learns relationships between patches from data. On smaller datasets, **pretraining** and image-specific architectural choices can be especially useful. The example in this notebook does not train a model or report accuracy.

## 12. Connection to the selected paper

The [ViTAEv2 paper](https://doi.org/10.1007/s11263-022-01739-w) builds on Vision Transformers by introducing image-specific **inductive biases**. Standard ViT uses patches and attention but has no built-in convolutional preference for nearby patterns or multiple visual scales.

The paper combines attention with a **Parallel Convolutional Module (PCM)** for local structure and a **Pyramid Reduction Module (PRM)** for multi-scale context. ViTAEv2 organizes these operations into stages that can provide features at different resolutions.

The progression is: **ViT represents patches and connects them with attention; ViTAEv2 adds convolutional pathways for local and multi-scale image information.**

### References

1. Zhang et al., [*ViTAEv2: Vision Transformer Advanced by Exploring Inductive Bias for Image Recognition and Beyond*](https://doi.org/10.1007/s11263-022-01739-w).
2. Dosovitskiy et al., [*An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*](https://arxiv.org/abs/2010.11929).

The code above illustrates ViT concepts and does not reproduce the paper's architecture or results.